# Module 7: Aggregate Functions for Analytics

**ALY 6420 | COUNT, SUM, AVG, MIN, MAX, GROUP BY, HAVING, and Aggregation with JOINs**

*Course Lecture Notes*


## Module 7

### From individual rows to analytical summaries

Earlier modules focused on retrieving, joining, and transforming rows. This module changes the level of analysis.

Instead of asking:

> What happened in this one transaction?

you begin asking:

> What is typical across many transactions?  
> How much activity occurred in each group?  
> Which groups are unusually large, small, profitable, or inactive?

Aggregate functions answer these questions by reducing many input rows to summary values. `GROUP BY` determines **which rows belong together**, and `HAVING` determines **which completed groups remain in the result**.


## Learning objectives

By the end of this lecture, you should be able to:

- apply `COUNT`, `SUM`, `AVG`, `MIN`, and `MAX` correctly
- explain how `NULL` affects each aggregate function
- distinguish `COUNT(*)`, `COUNT(column)`, and `COUNT(DISTINCT column)`
- write `GROUP BY` queries using one or more grouping columns
- explain the grain of an aggregated result
- distinguish row filtering with `WHERE` from group filtering with `HAVING`
- write conditional aggregates with `CASE` and PostgreSQL's `FILTER` clause
- group by expressions such as `DATE_TRUNC`
- diagnose one-to-many fan-out before aggregation
- use pre-aggregation to protect counts and sums
- interpret grouped output as evidence for a business question
- recognize when `ROLLUP` or `GROUPING SETS` can produce multi-level summaries


## Principal source and practice environment

### Principal resource

Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for Data Analytics* (4th ed.), **Chapter 8: Aggregating Data with GROUP BY**.

The textbook develops aggregation in a natural sequence:

1. calculate statistics over an entire dataset,
2. divide rows into meaningful groups,
3. calculate one summary per group,
4. filter grouped results with `HAVING`, and
5. extend grouping to multiple levels.

### Practice environment

Run the examples in DBeaver against the **Pagila** PostgreSQL database. The main tables used here are:

- `payment`
- `rental`
- `customer`
- `inventory`
- `film`
- `film_category`
- `category`


## A recurring question for this module

Before running any aggregate query, state the **result grain** in plain language.

Examples:

- one row for the entire `payment` table
- one row per customer
- one row per store and film rating
- one row per calendar month

If you cannot state what one output row represents, you are not yet ready to trust the aggregate.


# Part 1: What Aggregation Changes


## Row-level work versus aggregate work

A row-level query preserves individual observations:

```sql
SELECT payment_id,
       customer_id,
       amount
FROM payment;
```

Each result row still represents one payment.

An aggregate query changes that relationship:

```sql
SELECT SUM(amount) AS total_revenue
FROM payment;
```

Now thousands of payment rows are reduced to **one value**.

That reduction is the defining idea of aggregation.


## The input set matters

An aggregate function does not know what a "month," "customer," or "store" means on its own.

It simply summarizes the rows it receives.

Therefore, every aggregate question has two parts:

1. **Which rows should the function see?**
2. **At what grain should the result be returned?**

`WHERE`, JOIN conditions, and source tables determine the input rows.  
`GROUP BY` determines the output grain.


## Whole-table aggregation

Without `GROUP BY`, PostgreSQL treats all qualifying input rows as one group.

```sql
SELECT
    COUNT(*)              AS payment_rows,
    SUM(amount)           AS total_revenue,
    ROUND(AVG(amount), 2) AS avg_payment,
    MIN(amount)           AS smallest_payment,
    MAX(amount)           AS largest_payment
FROM payment;
```

Expected result grain:

> **one row for all qualifying payments**


## Quick check 1

**Think before opening the answer.**

A query contains `SUM(amount)` but no `GROUP BY`. How many rows should you normally expect?

<details>
<summary>Answer</summary>

One row, assuming the query has no other feature that changes the result shape. The entire qualifying input set is treated as one group.

</details>


## Aggregation is reduction, not sorting

`ORDER BY` changes presentation order.

`GROUP BY` changes the **logical structure** of the result.

Compare:

```sql
SELECT customer_id, amount
FROM payment
ORDER BY customer_id;
```

with:

```sql
SELECT customer_id, SUM(amount)
FROM payment
GROUP BY customer_id;
```

The first still has one row per payment.  
The second has one row per customer represented in `payment`.


# Part 2: The Core Aggregate Functions


## The five core aggregates

| Function | Main question |
|---|---|
| `COUNT` | How many? |
| `SUM` | How much in total? |
| `AVG` | What is the mean? |
| `MIN` | What is the smallest / earliest? |
| `MAX` | What is the largest / latest? |

Shan et al. also discuss additional statistical aggregates such as standard deviation and variance. In this module, the five functions above form the core foundation.


## `COUNT(*)`: count rows

```sql
SELECT COUNT(*) AS total_payments
FROM payment;
```

`COUNT(*)` counts **rows**, regardless of whether individual columns inside those rows contain `NULL`.

Use it when your business question is literally:

> How many records are in this input set?


## `COUNT(column)`: count non-NULL values

```sql
SELECT COUNT(return_date) AS returned_rentals
FROM rental;
```

`COUNT(return_date)` ignores rows where `return_date` is `NULL`.

This makes `COUNT(column)` useful for **data completeness** questions.

For example:

```sql
SELECT
    COUNT(*) AS all_rentals,
    COUNT(return_date) AS returned_rentals,
    COUNT(*) - COUNT(return_date) AS rentals_without_return_date
FROM rental;
```


## `COUNT(DISTINCT column)`: count unique non-NULL values

```sql
SELECT COUNT(DISTINCT customer_id) AS customers_with_payments
FROM payment;
```

This is a different question from `COUNT(*)`.

- `COUNT(*)` = how many payment rows?
- `COUNT(DISTINCT customer_id)` = how many different customers appear?

A common analytical error is using one when the business question requires the other.


## Quick check 2

**Think before opening the answer.**


A customer made 12 payments. In a query limited to that customer, what do these return?

```sql
COUNT(*)
COUNT(DISTINCT customer_id)
```


<details>
<summary>Answer</summary>

`COUNT(*)` returns 12 because there are 12 payment rows. `COUNT(DISTINCT customer_id)` returns 1 because all 12 rows belong to one unique customer.

</details>


## `SUM`: add numeric values

```sql
SELECT SUM(amount) AS total_revenue
FROM payment;
```

`SUM` answers additive questions such as:

- total revenue
- total units sold
- total hours
- total cost
- total claims paid

Before summing, ask whether the measure is truly additive at the current grain.


## `AVG`: calculate a mean

```sql
SELECT ROUND(AVG(amount), 2) AS avg_payment
FROM payment;
```

Conceptually:

\[
	ext{average} =
rac{	ext{sum of non-NULL values}}
{	ext{count of non-NULL values}}
\]

`AVG` silently excludes `NULL`. That behavior is correct only if it matches the meaning of missing data.


## `MIN` and `MAX`

```sql
SELECT
    MIN(amount) AS minimum_payment,
    MAX(amount) AS maximum_payment
FROM payment;
```

These functions work beyond numeric columns.

For example:

```sql
SELECT
    MIN(payment_date) AS first_payment,
    MAX(payment_date) AS latest_payment
FROM payment;
```

For dates, `MIN` means earliest and `MAX` means latest.


## Text values can also have a minimum and maximum

PostgreSQL can apply `MIN` and `MAX` to text according to the database's ordering rules.

```sql
SELECT
    MIN(last_name) AS alphabetically_first,
    MAX(last_name) AS alphabetically_last
FROM customer;
```

This is valid SQL, but always ask whether the resulting statistic has useful business meaning.


## Textbook enrichment: dispersion

Chapter 8 also introduces statistical aggregates such as `STDDEV`.

```sql
SELECT
    AVG(amount)    AS mean_payment,
    STDDEV(amount) AS payment_stddev
FROM payment;
```

The average describes the center. Standard deviation describes how widely values vary around that center.

This is not a primary Module 7 requirement, but it is useful preparation for later analytical work.


# Part 3: NULLs and Aggregation


## NULL behavior is silent

Most aggregates ignore `NULL` values without warning.

That means a query can execute perfectly and still answer a different question from the one you intended.

| Expression | What is counted / summarized |
|---|---|
| `COUNT(*)` | every row |
| `COUNT(col)` | only rows where `col` is non-NULL |
| `SUM(col)` | non-NULL values |
| `AVG(col)` | non-NULL values |
| `MIN(col)` | non-NULL values |
| `MAX(col)` | non-NULL values |


## Why `AVG` deserves extra attention

Suppose a metric contains:

```text
10, 20, NULL, NULL
```

Then:

```sql
AVG(metric)
```

calculates:

```text
(10 + 20) / 2 = 15
```

It does **not** divide by four.

Whether that is correct depends on what `NULL` means.


## Missing versus zero

These are different business meanings:

- `NULL` = value is unknown / not recorded
- `0` = value is known and equals zero

If missing values should logically behave as zero, make that rule explicit:

```sql
SELECT AVG(COALESCE(metric, 0))
FROM some_table;
```

Do not automatically replace every `NULL` with zero. That is a domain decision, not a formatting decision.


## Quick check 3

**Think before opening the answer.**

Why can `AVG(column)` be misleading when `NULL` means 'no activity' rather than 'unknown'?

<details>
<summary>Answer</summary>

Because `AVG(column)` drops NULL rows from the denominator. If those rows should count as zero activity, excluding them will usually make the average too high.

</details>


## Measure completeness directly

A useful profiling pattern is:

```sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(return_date) AS populated_return_dates,
    COUNT(*) - COUNT(return_date) AS missing_return_dates
FROM rental;
```

This uses aggregation not only to analyze business activity but also to assess data quality.


# Part 4: GROUP BY — One Summary per Group


## The GROUP BY model

The textbook describes `GROUP BY` as dividing a dataset into groups based on a key, then applying an aggregate function inside each group.

Think of three stages:

```text
input rows
   ↓
assign each row to a bucket
   ↓
aggregate inside each bucket
   ↓
one result row per bucket
```

The grouping key defines the output grain.


## First grouped query: payments by customer

```sql
SELECT
    customer_id,
    COUNT(*) AS num_payments,
    SUM(amount) AS total_paid
FROM payment
GROUP BY customer_id
ORDER BY total_paid DESC;
```

Result grain:

> **one row per customer_id appearing in payment**


## Predict before running

Ask:

- Is `customer_id` unique in `payment`? No.
- Will the output contain one row per payment? No.
- How many output rows should there be? Approximately the number of distinct customers represented in `payment`.

A useful validation query is:

```sql
SELECT COUNT(DISTINCT customer_id)
FROM payment;
```

That count should match the number of grouped result rows.


## The GROUP BY rule

Every selected expression must be either:

1. part of the grouping key, or
2. reduced by an aggregate function.

This is invalid:

```sql
SELECT customer_id,
       payment_date,
       SUM(amount)
FROM payment
GROUP BY customer_id;
```

Which `payment_date` should PostgreSQL show for a customer with many payments? There is no single defined answer.


## Valid alternatives

### Group at a finer grain

```sql
SELECT customer_id,
       payment_date,
       SUM(amount)
FROM payment
GROUP BY customer_id, payment_date;
```

### Or aggregate the date

```sql
SELECT customer_id,
       MAX(payment_date) AS latest_payment,
       SUM(amount) AS total_paid
FROM payment
GROUP BY customer_id;
```

These queries answer different questions because their result grains differ.


## Quick check 4

**Think before opening the answer.**


What is the grain of this result?

```sql
SELECT customer_id, staff_id, SUM(amount)
FROM payment
GROUP BY customer_id, staff_id;
```


<details>
<summary>Answer</summary>

One row per unique `(customer_id, staff_id)` combination.

</details>


## Group by more than one column

```sql
SELECT
    customer_id,
    staff_id,
    COUNT(*) AS payment_count,
    SUM(amount) AS total_amount
FROM payment
GROUP BY customer_id, staff_id
ORDER BY customer_id, staff_id;
```

Each unique combination becomes its own group.

Adding a grouping column usually makes the result grain **finer**.


## Group by an expression

The grouping key does not have to be a raw column.

```sql
SELECT
    DATE_TRUNC('month', payment_date) AS pay_month,
    COUNT(*) AS payment_count,
    SUM(amount) AS monthly_revenue
FROM payment
GROUP BY DATE_TRUNC('month', payment_date)
ORDER BY pay_month;
```

Now the grain is:

> **one row per calendar month**


## Bridge from Module 6

Module 6 introduced `DATE_TRUNC` as a transformation.

Module 7 uses the transformed value as a grouping key.

This pattern appears constantly in real reporting:

```sql
DATE_TRUNC('day', timestamp)
DATE_TRUNC('week', timestamp)
DATE_TRUNC('month', timestamp)
DATE_TRUNC('quarter', timestamp)
DATE_TRUNC('year', timestamp)
```

The transformation defines the time bucket; the aggregate summarizes activity inside it.


## Group by a business category

You can group by derived categories as well.

```sql
SELECT
    CASE
        WHEN amount < 2 THEN 'Low'
        WHEN amount < 6 THEN 'Medium'
        ELSE 'High'
    END AS payment_band,
    COUNT(*) AS payment_count,
    SUM(amount) AS revenue
FROM payment
GROUP BY
    CASE
        WHEN amount < 2 THEN 'Low'
        WHEN amount < 6 THEN 'Medium'
        ELSE 'High'
    END
ORDER BY payment_band;
```

This is where row-level transformation and aggregation meet.


# Part 5: WHERE and HAVING


## Two filters, two different stages

`WHERE` and `HAVING` both remove data, but they act on different objects.

### `WHERE`
Filters **input rows before grouping**.

### `HAVING`
Filters **completed groups after aggregation**.

That distinction follows SQL's logical execution order.


## `WHERE`: filter rows first

```sql
SELECT
    customer_id,
    SUM(amount) AS total_paid
FROM payment
WHERE payment_date >= DATE '2005-07-01'
GROUP BY customer_id;
```

Interpretation:

1. keep only payments on or after July 1,
2. group the remaining rows by customer,
3. sum each customer's retained payments.


## `HAVING`: filter groups

```sql
SELECT
    customer_id,
    SUM(amount) AS total_paid
FROM payment
GROUP BY customer_id
HAVING SUM(amount) > 150
ORDER BY total_paid DESC;
```

Interpretation:

1. group all payments by customer,
2. calculate each customer's total,
3. keep only groups whose total exceeds 150.


## Both filters in one query

```sql
SELECT
    customer_id,
    COUNT(*) AS payment_count,
    SUM(amount) AS total_paid
FROM payment
WHERE payment_date >= DATE '2005-07-01'
GROUP BY customer_id
HAVING SUM(amount) > 100
ORDER BY total_paid DESC;
```

`WHERE` changes **which rows enter the groups**.  
`HAVING` changes **which completed groups survive**.


## Why aggregates do not belong in WHERE

This is invalid:

```sql
SELECT customer_id,
       SUM(amount)
FROM payment
WHERE SUM(amount) > 100
GROUP BY customer_id;
```

At the time `WHERE` runs, the customer totals do not exist yet.

The aggregate condition belongs in `HAVING`.


## Quick check 5

**Think before opening the answer.**


You want customers who:

- made payments during July, and
- spent more than $100 during that filtered period.

Which condition belongs in `WHERE`, and which belongs in `HAVING`?


<details>
<summary>Answer</summary>

The July date condition belongs in `WHERE` because it filters individual payment rows. `SUM(amount) > 100` belongs in `HAVING` because it filters the completed customer groups.

</details>


## A useful diagnostic sentence

When choosing between `WHERE` and `HAVING`, finish one of these sentences:

> Remove this **row** because ...

or

> Remove this **group** because ...

If you are talking about a row, think `WHERE`.  
If you are talking about a summary of a group, think `HAVING`.


# Part 6: Conditional Aggregation


## One group, several conditional measures

Often you want several measures from the same grouped rows.

Example business question:

> For each store, how many payments were small, medium, and large?

One approach is `CASE` inside an aggregate.


## `SUM(CASE ...)` pattern

```sql
SELECT
    c.store_id,
    COUNT(*) AS total_payments,
    SUM(CASE WHEN p.amount < 2 THEN 1 ELSE 0 END) AS low_payments,
    SUM(CASE WHEN p.amount >= 6 THEN 1 ELSE 0 END) AS high_payments
FROM payment p
JOIN customer c
  ON p.customer_id = c.customer_id
GROUP BY c.store_id
ORDER BY c.store_id;
```

Each row contributes either `1` or `0` to the conditional sum.


## `COUNT(CASE ...)` pattern

```sql
SELECT
    c.store_id,
    COUNT(*) AS total_payments,
    COUNT(CASE WHEN p.amount < 2 THEN 1 END) AS low_payments,
    COUNT(CASE WHEN p.amount >= 6 THEN 1 END) AS high_payments
FROM payment p
JOIN customer c
  ON p.customer_id = c.customer_id
GROUP BY c.store_id;
```

Rows that do not meet the condition produce `NULL`, and `COUNT(expression)` ignores `NULL`.


## PostgreSQL `FILTER`

PostgreSQL provides a concise alternative:

```sql
SELECT
    c.store_id,
    COUNT(*) AS total_payments,
    COUNT(*) FILTER (WHERE p.amount < 2) AS low_payments,
    COUNT(*) FILTER (WHERE p.amount >= 6) AS high_payments
FROM payment p
JOIN customer c
  ON p.customer_id = c.customer_id
GROUP BY c.store_id;
```

The outer grouping stays the same. Each aggregate sees only rows matching its own `FILTER`.


## Why conditional aggregation matters

A dashboard often needs several metrics at the same grain:

```text
store_id | payments | low_value | high_value | revenue
```

Without conditional aggregation, you might write several queries and join them later.

Conditional aggregation can calculate all of those measures in one grouped result.


## Quick check 6

**Think before opening the answer.**

Does `FILTER (WHERE ...)` remove rows from the entire query?

<details>
<summary>Answer</summary>

No. It restricts the rows seen by that particular aggregate function. Other aggregates in the same SELECT can use different FILTER conditions or no FILTER at all.

</details>


# Part 7: Aggregation with JOINs — Grain and Fan-Out


## JOINs change the row set before aggregation

This is one of the most important ideas in analytical SQL.

SQL effectively sees:

```text
FROM / JOIN
   ↓
WHERE
   ↓
GROUP BY
   ↓
aggregates
```

So if a JOIN duplicates rows, the aggregate receives the duplicated result.


## A one-to-many relationship

In Pagila:

```text
customer 1 ─────< many payment rows
```

If one customer has 30 payments, joining `customer` to `payment` creates 30 joined rows for that customer.

That is not an error. It is the expected relational result.

The danger appears when you aggregate without understanding that new grain.


## Count the rows before and after a join

```sql
SELECT COUNT(*)
FROM customer;
```

Now:

```sql
SELECT COUNT(*)
FROM customer c
JOIN payment p
  ON c.customer_id = p.customer_id;
```

The second count is much larger because the joined result is at approximately **payment grain**, not customer grain.


## The classic wrong question

Suppose you want:

> How many customers belong to each store?

This query is correct:

```sql
SELECT store_id,
       COUNT(*) AS customers
FROM customer
GROUP BY store_id;
```

But if you unnecessarily join payment first:

```sql
SELECT c.store_id,
       COUNT(*) AS customers
FROM customer c
JOIN payment p
  ON c.customer_id = p.customer_id
GROUP BY c.store_id;
```

you are counting joined payment rows, not customers.


## One possible repair: count distinct customers

```sql
SELECT c.store_id,
       COUNT(DISTINCT c.customer_id) AS customers
FROM customer c
JOIN payment p
  ON c.customer_id = p.customer_id
GROUP BY c.store_id;
```

This can be appropriate when the intended population is:

> customers who have at least one matching payment

But do not automatically add `DISTINCT` to every inflated query. First determine why the duplicates exist.


## Safer design: aggregate before joining

If you need a customer-level payment summary, create it first:

```sql
WITH customer_totals AS (
    SELECT
        customer_id,
        COUNT(*) AS payment_count,
        SUM(amount) AS total_paid
    FROM payment
    GROUP BY customer_id
)
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    ct.payment_count,
    ct.total_paid
FROM customer c
LEFT JOIN customer_totals ct
  ON c.customer_id = ct.customer_id
ORDER BY ct.total_paid DESC NULLS LAST;
```

The CTE has one row per customer, so joining it to `customer` preserves customer grain.


## Pre-aggregation is a grain-control technique

The reason the previous pattern works is not that CTEs are magical.

It works because the CTE deliberately converts:

```text
many payment rows per customer
```

into:

```text
one summary row per customer
```

before the join.

You can use a derived table instead of a CTE and get the same logical protection.


## Quick check 7

**Think before opening the answer.**


A query joins `film` → `inventory` → `rental` and then runs `COUNT(*) GROUP BY film_id`.

What does the count represent?


<details>
<summary>Answer</summary>

It represents the number of joined rental rows per film, assuming the join path is correct. It does not represent the number of films or the number of inventory copies.

</details>


## Fan-out can also inflate SUM

Imagine a table with one row per order containing `order_total`, joined to a line-item table with five rows for the order.

After the join, that same `order_total` may appear five times.

Then:

```sql
SUM(order_total)
```

would count the order total five times.

The general lesson:

> Never sum a measure until you know the grain at which that measure is valid.


# Part 8: Business Questions with Pagila


## Example 1: revenue by store

```sql
SELECT
    c.store_id,
    COUNT(*) AS payment_count,
    ROUND(SUM(p.amount), 2) AS revenue,
    ROUND(AVG(p.amount), 2) AS avg_payment
FROM payment p
JOIN customer c
  ON p.customer_id = c.customer_id
GROUP BY c.store_id
ORDER BY c.store_id;
```

Result grain:

> one row per store


## Interpreting the result

Do not stop at "store 1 has X and store 2 has Y."

A useful interpretation separates:

- **volume** — payment count
- **value** — total revenue
- **typical transaction size** — average payment

A store can lead in total revenue because it has more transactions, larger average payments, or both.


## Example 2: rental activity by film rating

```sql
SELECT
    f.rating,
    COUNT(*) AS rentals
FROM rental r
JOIN inventory i
  ON r.inventory_id = i.inventory_id
JOIN film f
  ON i.film_id = f.film_id
GROUP BY f.rating
ORDER BY rentals DESC;
```

Result grain:

> one row per film rating


## Example 3: revenue by month

```sql
SELECT
    DATE_TRUNC('month', payment_date) AS month,
    COUNT(*) AS payments,
    ROUND(SUM(amount), 2) AS revenue,
    ROUND(AVG(amount), 2) AS avg_payment
FROM payment
GROUP BY DATE_TRUNC('month', payment_date)
ORDER BY month;
```

This supports time-series questions such as:

- Which month produced the most revenue?
- Did transaction count and average payment move together?
- Was revenue growth driven by volume or transaction size?


## Example 4: categories with high rental volume

```sql
SELECT
    cat.name AS category,
    COUNT(*) AS rental_count
FROM rental r
JOIN inventory i
  ON r.inventory_id = i.inventory_id
JOIN film_category fc
  ON i.film_id = fc.film_id
JOIN category cat
  ON fc.category_id = cat.category_id
GROUP BY cat.name
HAVING COUNT(*) >= 900
ORDER BY rental_count DESC;
```

`HAVING` is appropriate because the condition is based on the completed category count.


## Example 5: customers above a spending threshold

```sql
SELECT
    p.customer_id,
    c.first_name,
    c.last_name,
    COUNT(*) AS payment_count,
    ROUND(SUM(p.amount), 2) AS total_paid
FROM payment p
JOIN customer c
  ON p.customer_id = c.customer_id
GROUP BY p.customer_id, c.first_name, c.last_name
HAVING SUM(p.amount) > 150
ORDER BY total_paid DESC;
```

The grouped result can support loyalty, retention, or customer-value analysis.


# Part 9: Multi-Level Summaries


## Why one grouping level is sometimes not enough

A manager may want:

- revenue for each store,
- revenue for each store by another dimension, and
- a company-wide total.

You can write separate queries and combine them, but SQL has grouping extensions designed for this purpose.


## `ROLLUP`

```sql
SELECT
    c.store_id,
    ROUND(SUM(p.amount), 2) AS revenue
FROM payment p
JOIN customer c
  ON p.customer_id = c.customer_id
GROUP BY ROLLUP(c.store_id)
ORDER BY c.store_id NULLS LAST;
```

The result includes:

- one row per store
- one grand-total row


## Interpreting the NULL subtotal row

With `ROLLUP`, a `NULL` grouping value can represent a generated subtotal or grand total.

That is different from an ordinary data row whose grouping column is actually `NULL`.

For more complex reports, PostgreSQL's `GROUPING()` function can help distinguish generated summary rows from real NULL values.


## `GROUPING SETS`

The textbook shows that different grouping levels can be requested explicitly.

A Pagila-style example:

```sql
SELECT
    c.store_id,
    f.rating,
    COUNT(*) AS rentals
FROM rental r
JOIN customer c
  ON r.customer_id = c.customer_id
JOIN inventory i
  ON r.inventory_id = i.inventory_id
JOIN film f
  ON i.film_id = f.film_id
GROUP BY GROUPING SETS (
    (c.store_id, f.rating),
    (c.store_id),
    ()
);
```

This requests:

1. store + rating detail,
2. store subtotals,
3. a grand total.


## Scope note

`ROLLUP` and `GROUPING SETS` are useful extensions, but your first priority is to master:

- core aggregate functions,
- grain,
- `GROUP BY`,
- `WHERE` versus `HAVING`, and
- fan-out control.

Advanced grouping is much easier once those foundations are stable.


# Part 10: Debugging Aggregate Queries


## Error 1: selected column is neither grouped nor aggregated

Problem:

```sql
SELECT customer_id,
       payment_date,
       SUM(amount)
FROM payment
GROUP BY customer_id;
```

PostgreSQL rejects the query because `payment_date` has no single defined value for a customer group.

Fix the grain, not just the syntax.


## Error 2: aggregate in WHERE

Problem:

```sql
SELECT customer_id, SUM(amount)
FROM payment
WHERE SUM(amount) > 100
GROUP BY customer_id;
```

Fix:

```sql
SELECT customer_id, SUM(amount)
FROM payment
GROUP BY customer_id
HAVING SUM(amount) > 100;
```


## Error 3: `COUNT(column)` undercounts

If a nullable column contains missing values:

```sql
COUNT(column)
```

does not count those rows.

If your question is "how many rows?", use:

```sql
COUNT(*)
```


## Error 4: DISTINCT hides a grain problem

You notice too many rows and write:

```sql
SELECT DISTINCT ...
```

The output now looks reasonable.

That does not prove the logic is correct.

Before using `DISTINCT`, inspect:

```sql
SELECT COUNT(*) ...
SELECT COUNT(DISTINCT key) ...
```

and identify the relationship that produced repeated rows.


## Error 5: GROUP BY too coarse

Suppose the business question is:

> Monthly revenue per store.

This is too coarse:

```sql
GROUP BY DATE_TRUNC('month', payment_date)
```

because it combines both stores.

The intended grain requires both dimensions:

```sql
GROUP BY
    c.store_id,
    DATE_TRUNC('month', p.payment_date)
```


## Error 6: GROUP BY too fine

Suppose the question is:

> Total revenue per customer.

If you group by:

```sql
customer_id, payment_id
```

then each payment becomes its own group.

The aggregate cannot meaningfully collapse the customer's rows.

The grouping columns define the answer, so every extra grouping column deserves scrutiny.


## A reliable debugging sequence

When an aggregate result looks wrong:

1. state the intended output grain
2. inspect the base table grain
3. inspect JOIN cardinalities
4. count rows before grouping
5. count distinct business keys
6. remove aggregates temporarily and inspect sample rows
7. rebuild the grouping step
8. apply `HAVING` only after validating the groups


## Quick check 8

**Think before opening the answer.**

Which is usually safer when a one-to-many JOIN threatens to inflate a measure: add `DISTINCT` everywhere, or aggregate to the intended grain before joining?

<details>
<summary>Answer</summary>

Aggregate to the intended grain before joining. `DISTINCT` can be appropriate in specific cases, but it should not be used as a generic patch for misunderstood fan-out.

</details>


# Part 11: Guided Practice


## Practice 1: whole-table statistics

Write one query against `payment` that returns:

- total number of payment rows
- total revenue
- average payment
- minimum payment
- maximum payment

Before running it, predict the number of result rows.


<details>
<summary>Solution</summary>

```sql
SELECT
    COUNT(*) AS payment_count,
    SUM(amount) AS revenue,
    AVG(amount) AS avg_payment,
    MIN(amount) AS min_payment,
    MAX(amount) AS max_payment
FROM payment;
```

The result has one row because there is no `GROUP BY`.

</details>


## Practice 2: monthly revenue

Return one row per payment month with:

- payment count
- total revenue
- average payment

Sort chronologically.


<details>
<summary>Solution</summary>

```sql
SELECT
    DATE_TRUNC('month', payment_date) AS payment_month,
    COUNT(*) AS payment_count,
    ROUND(SUM(amount), 2) AS revenue,
    ROUND(AVG(amount), 2) AS avg_payment
FROM payment
GROUP BY DATE_TRUNC('month', payment_date)
ORDER BY payment_month;
```

</details>


## Practice 3: high-value customer groups

Return customers whose total payments exceed 150.

Include:

- customer ID
- first name
- last name
- number of payments
- total paid


<details>
<summary>Solution</summary>

```sql
SELECT
    p.customer_id,
    c.first_name,
    c.last_name,
    COUNT(*) AS payment_count,
    ROUND(SUM(p.amount), 2) AS total_paid
FROM payment p
JOIN customer c
  ON p.customer_id = c.customer_id
GROUP BY p.customer_id, c.first_name, c.last_name
HAVING SUM(p.amount) > 150
ORDER BY total_paid DESC;
```

</details>


## Practice 4: store + rating grain

Write a query that returns rental count for each combination of:

- customer store
- film rating

State the output grain before writing the SQL.


<details>
<summary>Solution</summary>

Grain: **one row per `(store_id, rating)` combination**.

```sql
SELECT
    c.store_id,
    f.rating,
    COUNT(*) AS rental_count
FROM rental r
JOIN customer c
  ON r.customer_id = c.customer_id
JOIN inventory i
  ON r.inventory_id = i.inventory_id
JOIN film f
  ON i.film_id = f.film_id
GROUP BY c.store_id, f.rating
ORDER BY c.store_id, f.rating;
```

</details>


## Practice 5: conditional aggregation

For each store, return:

- all payment rows
- payments under 2
- payments at least 6
- total revenue

Use `FILTER`.


<details>
<summary>Solution</summary>

```sql
SELECT
    c.store_id,
    COUNT(*) AS all_payments,
    COUNT(*) FILTER (WHERE p.amount < 2) AS under_2,
    COUNT(*) FILTER (WHERE p.amount >= 6) AS at_least_6,
    ROUND(SUM(p.amount), 2) AS revenue
FROM payment p
JOIN customer c
  ON p.customer_id = c.customer_id
GROUP BY c.store_id
ORDER BY c.store_id;
```

</details>


## Practice 6: fan-out diagnosis

A colleague writes:

```sql
SELECT c.store_id,
       COUNT(*) AS customer_count
FROM customer c
JOIN payment p
  ON c.customer_id = p.customer_id
GROUP BY c.store_id;
```

Their goal is to count customers per store.

Explain why the result is inflated and write two valid alternatives.


<details>
<summary>Solution</summary>

The JOIN creates one row per matching payment, so `COUNT(*)` counts payment-grain rows.

Simplest alternative:

```sql
SELECT store_id,
       COUNT(*) AS customer_count
FROM customer
GROUP BY store_id;
```

If the business question specifically means customers with a payment:

```sql
SELECT c.store_id,
       COUNT(DISTINCT c.customer_id) AS customer_count
FROM customer c
JOIN payment p
  ON c.customer_id = p.customer_id
GROUP BY c.store_id;
```

</details>


# Part 12: Real-World Analytical Patterns


## Sales analytics

A retailer commonly groups transaction data by:

- store
- product
- customer segment
- day / week / month
- channel

Typical measures:

- `COUNT(*)` transactions
- `SUM(revenue)` sales
- `AVG(order_value)` typical transaction
- `COUNT(DISTINCT customer_id)` active customers


## Healthcare analytics

Examples include:

- visits per clinic
- average wait time by department
- patients with more than N visits
- monthly claim totals
- missing follow-up rates

The same SQL foundations apply, but the meaning of a row and of `NULL` becomes especially important.


## Operations analytics

Examples:

- tickets per support team
- average resolution time
- shipments per warehouse
- total delay hours
- categories exceeding a threshold

`GROUP BY` creates the management level of interest; `HAVING` highlights groups needing attention.


## Finance analytics

Examples:

- total expense by cost center
- monthly transaction volume
- average claim amount by product
- accounts above a risk threshold
- minimum and maximum daily balance

The challenge is often not the aggregate function itself. It is maintaining the correct grain through joins.


# Part 13: From SQL Output to Analytical Interpretation


## A correct query is only the first step

Suppose your grouped result shows:

```text
store_id | payments | revenue | avg_payment
```

A useful business interpretation should ask:

- Which store leads on volume?
- Which leads on revenue?
- Is the revenue difference explained by payment count, average payment, or both?
- Is the difference large enough to matter operationally?
- What additional dimension would help explain the result?


## Avoid causal language from descriptive aggregates

If store 1 has higher revenue, the query shows an **association in the observed data**.

It does not prove that the store location caused higher revenue.

Aggregation describes patterns. Causal conclusions require stronger design and evidence.


## Pair totals with denominators

A total alone can be misleading.

For example:

> Store A has more revenue than Store B.

Useful follow-up measures include:

- number of transactions
- number of unique customers
- average payment
- active days
- revenue per customer

Good analytical reporting combines scale and rate measures.


# Part 14: AI Critique and Query Review


## Critiquing generated aggregate SQL

AI-generated SQL often looks syntactically polished while containing grain errors.

When reviewing generated aggregation code, check:

1. What does one source row represent?
2. What does one output row represent?
3. Can any JOIN multiply rows?
4. Is the measure valid at the post-JOIN grain?
5. Is `COUNT(*)` answering a row-count question?
6. Are nullable columns handled intentionally?
7. Are aggregate filters in `HAVING` rather than `WHERE`?
8. Does the grouping level match the business question?


## AI critique exercise

A model suggests:

```sql
SELECT
    c.store_id,
    COUNT(*) AS customers,
    SUM(p.amount) AS revenue
FROM customer c
JOIN payment p
  ON c.customer_id = p.customer_id
GROUP BY c.store_id;
```

It explains:

> "This query counts the number of customers in each store and calculates their revenue."

Identify what is correct and what is misleading.


<details>
<summary>Critique</summary>

`SUM(p.amount)` can correctly calculate payment revenue per store because the joined rows are at payment grain and each payment amount appears once.

`COUNT(*) AS customers` is misleading. After the one-to-many join, `COUNT(*)` counts joined payment rows, not customers.

If the desired customer measure is customers with at least one payment, use:

```sql
COUNT(DISTINCT c.customer_id)
```

Or calculate customer count directly from `customer` without the payment join.

</details>


## Verification should be visible

When using AI for SQL assistance, do not validate by saying:

> "The query ran."

A stronger verification process includes:

```sql
SELECT COUNT(*) ...
SELECT COUNT(DISTINCT key) ...
```

and small sample inspections that confirm the intended grain.

Execution success is not evidence of analytical correctness.


# Part 15: Knowledge Checks


## Knowledge check 1

**Think before opening the answer.**

Which aggregate counts every row even if some columns are NULL?

<details>
<summary>Answer</summary>

`COUNT(*)`.

</details>


## Knowledge check 2

**Think before opening the answer.**

What does `COUNT(email)` measure if `email` is nullable?

<details>
<summary>Answer</summary>

The number of rows whose `email` value is not NULL.

</details>


## Knowledge check 3

**Think before opening the answer.**

What does adding a second column to `GROUP BY` usually do to the result grain?

<details>
<summary>Answer</summary>

It makes the grain finer by creating one group per unique combination of both grouping columns.

</details>


## Knowledge check 4

**Think before opening the answer.**

Can `HAVING` appear without `WHERE`?

<details>
<summary>Answer</summary>

Yes. `HAVING` can filter grouped results even when no row-level WHERE filter is needed.

</details>


## Knowledge check 5

**Think before opening the answer.**

Can `WHERE` reference `SUM(amount)` in the same query block?

<details>
<summary>Answer</summary>

No. WHERE logically runs before grouping and aggregation. Use HAVING for aggregate conditions.

</details>


## Knowledge check 6

**Think before opening the answer.**

Why might `COUNT(DISTINCT customer_id)` be much smaller than `COUNT(*)` in `payment`?

<details>
<summary>Answer</summary>

Because one customer can have many payment rows. COUNT(*) measures payment rows; COUNT(DISTINCT customer_id) measures unique customers.

</details>


## Knowledge check 7

**Think before opening the answer.**

What is fan-out?

<details>
<summary>Answer</summary>

Fan-out is row multiplication caused by joining one row to multiple matching rows, such as one customer to many payments. It can inflate aggregates when the post-join grain is misunderstood.

</details>


## Knowledge check 8

**Think before opening the answer.**

What is the safest first question when an aggregate looks too large?

<details>
<summary>Answer</summary>

Ask what one row represents after all JOINs and before GROUP BY.

</details>


# Part 16: Concept Map


## Module 7 concept map

```text
RAW / TRANSFORMED ROWS
        |
        v
FROM + JOIN
        |
        |  establish the input row set
        |  watch cardinality / fan-out
        v
WHERE
        |
        |  remove individual rows
        v
GROUP BY
        |
        |  assign rows to buckets
        |  defines result grain
        v
AGGREGATE FUNCTIONS
        |
        |  COUNT / SUM / AVG / MIN / MAX
        |  one value per group
        v
HAVING
        |
        |  remove completed groups
        v
SELECTED SUMMARY ROWS
        |
        v
ORDER BY
```


## A second map: choose the function by the question

```text
How many rows?  -----------------> COUNT(*)

How many known values? ----------> COUNT(column)

How many unique values? ---------> COUNT(DISTINCT column)

How much altogether? ------------> SUM(column)

What is typical on average? -----> AVG(column)

What is smallest / earliest? ----> MIN(column)

What is largest / latest? -------> MAX(column)

Per customer/store/month/etc.? --> GROUP BY

Only groups above a threshold? --> HAVING
```


# Part 17: Lab 6 Part I Readiness


## Before starting the lab

You should be able to do each of these without copying a template:

- calculate whole-table aggregates
- explain `COUNT(*)` versus `COUNT(column)`
- explain how `AVG` treats `NULL`
- group by one column
- group by multiple columns
- group by a date expression
- sort by an aggregate result
- use `WHERE` before grouping
- use `HAVING` after grouping
- write conditional counts
- trace the grain through a multi-table JOIN
- diagnose inflated counts and sums


## Suggested validation habit for every lab query

Write a one-line comment above your query:

```sql
-- Grain: one row per customer
```

or:

```sql
-- Grain: one row per store per month
```

Then inspect whether your `GROUP BY` columns actually create that grain.

This small habit prevents many aggregation errors.


## Lab-style mini challenge

Business question:

> Which film categories generated the most rental activity, and keep only categories with at least 900 rentals?

Requirements:

- join the necessary Pagila tables
- group by category name
- use `COUNT`
- use `HAVING`
- sort from highest to lowest


<details>
<summary>One solution</summary>

```sql
SELECT
    cat.name AS category,
    COUNT(*) AS rental_count
FROM rental r
JOIN inventory i
  ON r.inventory_id = i.inventory_id
JOIN film_category fc
  ON i.film_id = fc.film_id
JOIN category cat
  ON fc.category_id = cat.category_id
GROUP BY cat.name
HAVING COUNT(*) >= 900
ORDER BY rental_count DESC;
```

Grain: one row per retained category.

</details>


# Part 18: Discussion Preparation


## Designing an analytical question

A good Module 7 business question has:

1. a measurable event or amount,
2. a meaningful grouping dimension, and
3. a reason the comparison matters.

Examples:

- How does rental volume differ by film rating?
- Which month generated the most payment revenue?
- Which film categories have above-average rental volume?
- How do stores differ in payment volume and average payment size?


## Explain results to a non-technical colleague

A strong interpretation does not describe SQL syntax.

Instead of:

> "I grouped by store_id and used SUM."

write:

> "Store 2 generated slightly more revenue during the observed period. The transaction counts were similar, so the difference appears to come partly from average payment size rather than volume alone."

Use the numbers from your own query, and avoid claiming causation.


## Peer-review questions

When reviewing a classmate's aggregate query, ask:

- Does the grouping grain match the business question?
- Is any JOIN multiplying rows?
- Would `COUNT(DISTINCT ...)` answer a more relevant question?
- Does a `HAVING` threshold reveal a useful subset?
- Could a second aggregate explain the first one?
- Is the interpretation supported by the result rather than assumed?


# Part 19: Module Summary


## Key takeaways

- Aggregate functions reduce many rows to summary values.
- Without `GROUP BY`, the qualifying input rows form one group.
- `COUNT(*)` counts rows; `COUNT(column)` counts non-NULL values; `COUNT(DISTINCT column)` counts unique non-NULL values.
- `SUM`, `AVG`, `MIN`, and `MAX` ignore `NULL`.
- `GROUP BY` defines the output grain by creating one group per unique key combination.
- Every selected non-aggregate expression must be compatible with the grouping grain.
- `WHERE` filters input rows before grouping.
- `HAVING` filters completed groups after aggregation.
- Conditional aggregation lets several measures be calculated at the same grain.
- JOIN cardinality must be understood before aggregation.
- Pre-aggregating to the intended grain is a reliable way to control fan-out.
- `ROLLUP` and `GROUPING SETS` extend grouping to multi-level summaries.


## The question to carry forward

Whenever you write an aggregate query, ask:

> **What exactly does one output row represent, and did every input row contribute exactly as intended?**

That question is the bridge from SQL that merely runs to SQL that produces trustworthy analytics.


## Up next: Module 8

`GROUP BY` summarizes a group by collapsing its rows into one output row.

Window functions solve a different problem:

> How can you calculate group-level or ordered statistics **without losing the original rows**?

Module 8 introduces:

- `OVER()`
- `PARTITION BY`
- window `ORDER BY`
- frame specifications
- ranking
- running totals
- moving calculations

The distinction between **grouping** and **partitioning** will be central.


## References

Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for data analytics: Analyze data effectively, uncover insights and master advanced SQL for real-world applications* (4th ed.). Packt Publishing.

PostgreSQL Global Development Group. (n.d.). *Aggregate functions*. PostgreSQL 16 Documentation.

PostgreSQL Global Development Group. (n.d.). *Table expressions*. PostgreSQL 16 Documentation.

Neon. (n.d.). *PostgreSQL aggregate functions*.
